In [ ]:
!pip install datasets transformers torch scikit-learn gensim nltk beautifulsoup4

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import re
from bs4 import BeautifulSoup
import warnings
import time
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Load Dataset
print("Loading IMDB dataset...")
dataset = load_dataset('imdb')

# Split Dataset
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")
print(train_df.head())

# Preprocessing Function
def preprocess_text(text):
    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = BeautifulSoup(text, 'html.parser').get_text()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization (for Word2Vec)
    tokens = word_tokenize(text)

    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words and len(token) > 2]

    return ' '.join(tokens), tokens

# Apply Preprocessing
print("Preprocessing data...")
start_time = time.time()
train_df['processed_text'] = train_df['text'].apply(lambda x: preprocess_text(x)[0])
train_df['tokens'] = train_df['text'].apply(lambda x: preprocess_text(x)[1])
test_df['processed_text'] = test_df['text'].apply(lambda x: preprocess_text(x)[0])
test_df['tokens'] = test_df['text'].apply(lambda x: preprocess_text(x)[1])
preprocess_time = time.time() - start_time
print(f"Preprocessing completed in {preprocess_time/60:.2f} minutes.")

# Results storage
results = []

# Method 1: TF-IDF
print("\n--- TF-IDF ---")
start_time = time.time()
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(train_df['processed_text'])
X_test_tfidf = vectorizer.transform(test_df['processed_text'])

clf_tfidf = LogisticRegression(max_iter=200, random_state=42)
clf_tfidf.fit(X_train_tfidf, train_df['label'])
preds_tfidf = clf_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(test_df['label'], preds_tfidf)
prec_tfidf = precision_score(test_df['label'], preds_tfidf)
rec_tfidf = recall_score(test_df['label'], preds_tfidf)
f1_tfidf = f1_score(test_df['label'], preds_tfidf)

results.append({
    'Method': 'TF-IDF',
    'Accuracy': acc_tfidf,
    'Precision': prec_tfidf,
    'Recall': rec_tfidf,
    'F1': f1_tfidf
})

tfidf_time = time.time() - start_time
print(f'TF-IDF Accuracy: {acc_tfidf:.4f} (Time: {tfidf_time/60:.2f} min)')

# Method 2: Word2Vec Embeddings
print("\n--- Word2Vec ---")
start_time = time.time()
sentences = train_df['tokens'].tolist()
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=1, workers=4)

def document_vector(doc_tokens, model, vector_size=100):
    vectors = [model.wv[word] for word in doc_tokens if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(vector_size)

X_train_w2v = np.array([document_vector(tokens, w2v_model) for tokens in train_df['tokens']])
X_test_w2v = np.array([document_vector(tokens, w2v_model) for tokens in test_df['tokens']])

clf_w2v = LogisticRegression(max_iter=200, random_state=42)
clf_w2v.fit(X_train_w2v, train_df['label'])
preds_w2v = clf_w2v.predict(X_test_w2v)

acc_w2v = accuracy_score(test_df['label'], preds_w2v)
prec_w2v = precision_score(test_df['label'], preds_w2v)
rec_w2v = recall_score(test_df['label'], preds_w2v)
f1_w2v = f1_score(test_df['label'], preds_w2v)

results.append({
    'Method': 'Word2Vec',
    'Accuracy': acc_w2v,
    'Precision': prec_w2v,
    'Recall': rec_w2v,
    'F1': f1_w2v
})

w2v_time = time.time() - start_time
print(f'Word2Vec Accuracy: {acc_w2v:.4f} (Time: {w2v_time/60:.2f} min)')

# Method 3: BERT Embeddings (using DistilBERT)
print("\n--- BERT ---")
start_time = time.time()
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
bert_model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model.to(device)
print(f"Using device: {device}")

def get_bert_embedding(text, tokenizer, model, max_length=256):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().squeeze()
    return embedding

# Compute Embeddings
print("Computing BERT embeddings for train...")
X_train_bert = np.array([get_bert_embedding(text, tokenizer, bert_model) for text in train_df['processed_text']])
print("Computing BERT embeddings for test...")
X_test_bert = np.array([get_bert_embedding(text, tokenizer, bert_model) for text in test_df['processed_text']])

clf_bert = LogisticRegression(max_iter=200, random_state=42)
clf_bert.fit(X_train_bert, train_df['label'])
preds_bert = clf_bert.predict(X_test_bert)

acc_bert = accuracy_score(test_df['label'], preds_bert)
prec_bert = precision_score(test_df['label'], preds_bert)
rec_bert = recall_score(test_df['label'], preds_bert)
f1_bert = f1_score(test_df['label'], preds_bert)

results.append({
    'Method': 'BERT',
    'Accuracy': acc_bert,
    'Precision': prec_bert,
    'Recall': rec_bert,
    'F1': f1_bert
})

bert_time = time.time() - start_time
print(f'BERT Embedding Accuracy: {acc_bert:.4f} (Time: {bert_time/60:.2f} min)')

# Comparison
print("\n--- Results Table ---")
results_df = pd.DataFrame(results)
print(results_df.round(4))